In [128]:
import argparse

import torch
import numpy as np

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud, get_color, draw_geometries
from geotransformer.utils.registration import compute_registration_error

from config import make_cfg
from model import create_model

import open3d as o3d




In [129]:
# Specify the desired GPU index (e.g., system GPU 1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [130]:
WEIGHTS = "../../output/geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn/snapshots/epoch-40.pth.tar"


In [131]:
cfg = make_cfg()
model = create_model(cfg).cuda()
state_dict = torch.load(WEIGHTS)
model.load_state_dict(state_dict["model"])

<All keys matched successfully>

In [132]:
REF_NUM = 25

In [133]:
SRC_FILE = f"../../data/faces/demo/src_{REF_NUM}.npy"
REF_FILE = f"../../data/faces/demo/ref_{REF_NUM}.npy"
GT_FILE = f"../../data/faces/demo/gt_{REF_NUM}.npy"
MORPHED_FULL_FILE = f"../../data/faces/demo/morphed_full_{REF_NUM}.npy"

In [134]:
def load_data():
    src_points = np.load(SRC_FILE)
    ref_points = np.load(REF_FILE)
    morphed_full_points = np.load(MORPHED_FULL_FILE)
    src_feats = np.ones_like(src_points[:, :1])
    ref_feats = np.ones_like(ref_points[:, :1])

    data_dict = {
        "ref_points": ref_points.astype(np.float32),
        "src_points": src_points.astype(np.float32),
        "ref_feats": ref_feats.astype(np.float32),
        "src_feats": src_feats.astype(np.float32),
        "morphed_full": morphed_full_points.astype(np.float32),
        "gt_z": np.zeros((32, 100), dtype=np.float32) 
    }

    if GT_FILE is not None:
        transform = np.load(GT_FILE)
        data_dict["transform"] = transform.astype(np.float32)

    return data_dict

def open3d_webrtc_draw(geometries):
    o3d.visualization.draw(geometries)
   

    
   

In [135]:
data_dict = load_data()

In [136]:
data_dict.keys()

dict_keys(['ref_points', 'src_points', 'ref_feats', 'src_feats', 'morphed_full', 'gt_z', 'transform'])

In [137]:

# prepare data
neighbor_limits = [38, 36, 36, 38]  # default setting in 3DMatch
data_dict = registration_collate_fn_stack_mode(
    [data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
)


# prediction
data_dict = to_cuda(data_dict)
output_dict = model(data_dict)
data_dict = release_cuda(data_dict)
output_dict = release_cuda(output_dict)

# get results
ref_points = output_dict["ref_points"]
src_points = output_dict["src_points"]
estimated_transform = output_dict["estimated_transform"]
transform = data_dict["transform"]

# visualization
ref_pcd = make_open3d_point_cloud(ref_points)
ref_pcd.estimate_normals()
ref_pcd.paint_uniform_color(get_color("custom_blue"))
src_pcd = make_open3d_point_cloud(src_points)
src_pcd.estimate_normals()
src_pcd.paint_uniform_color(get_color("custom_yellow"))
o3d.visualization.draw_plotly([ref_pcd, src_pcd])


In [138]:
estimated_t_src_pcd = src_pcd.transform(estimated_transform)
o3d.visualization.draw_plotly([ref_pcd, estimated_t_src_pcd])

# compute error
rre, rte = compute_registration_error(transform, estimated_transform)
print(f"RRE(deg): {rre:.3f}, RTE(m): {rte:.3f}")

RRE(deg): 0.635, RTE(m): 0.013


In [139]:
new_src_pcd = make_open3d_point_cloud(src_points)
new_src_pcd.transform(transform)
o3d.visualization.draw_plotly([estimated_t_src_pcd, new_src_pcd])

In [140]:
o3d.visualization.draw_plotly([ref_pcd, new_src_pcd])

In [141]:
# %%
# --- Visualize Predicted Morphed Shape vs. GT PCA Reconstruction ---

# 1. Extract the PREDICTED Morphed Shape from the model output
# This is generated using the predicted z_coefficients
pred_morphed_data = output_dict["morphed_full"]

if torch.is_tensor(pred_morphed_data):
    pred_morphed_data = pred_morphed_data.detach().cpu().numpy()
if pred_morphed_data.ndim == 3:
    pred_morphed_data = pred_morphed_data.squeeze(0)

pred_morphed_pcd = o3d.geometry.PointCloud()
pred_morphed_pcd.points = o3d.utility.Vector3dVector(pred_morphed_data)
pred_morphed_pcd.estimate_normals()
pred_morphed_pcd.paint_uniform_color([0.0, 1.0, 0.0]) # Green = Predicted Model Output

# 2. Extract the GT PCA Reconstruction from the model output
# This is generated using the true gt_z coefficients (recon_gt_points)
recon_gt_data = output_dict["recon_gt_points"]
if torch.is_tensor(recon_gt_data):
    recon_gt_data = recon_gt_data.detach().cpu().numpy()
if recon_gt_data.ndim == 3:
    recon_gt_data = recon_gt_data.squeeze(0)

recon_gt_pcd = o3d.geometry.PointCloud()
recon_gt_pcd.points = o3d.utility.Vector3dVector(recon_gt_data)
recon_gt_pcd.estimate_normals()
recon_gt_pcd.paint_uniform_color([0.0, 0.0, 1.0]) # Blue = GT PCA Reconstruction

print("Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)")
o3d.visualization.draw_plotly([pred_morphed_pcd, recon_gt_pcd])

# Optional side-by-side view (uncomment if you want to see them next to each other):
# recon_gt_pcd_shifted = copy.deepcopy(recon_gt_pcd).translate([0.5, 0, 0])
# o3d.visualization.draw_plotly([pred_morphed_pcd, recon_gt_pcd_shifted])
# %%

Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)


In [142]:
estimated_t_src_pcd.paint_uniform_color([0, 0, 0.5])
o3d.visualization.draw_plotly([pred_morphed_pcd, estimated_t_src_pcd])

In [143]:
# %%
import open3d as o3d
# Load and visualize the external .ply file
ply_path = "3099.ply"  # Ensure the path to your file is correct

try:
    # Load the point cloud
    ply_pcd = o3d.io.read_point_cloud(ply_path)
    
    if not ply_pcd.has_points():
        print(f"Failed to load {ply_path} or file is empty.")
    else:
        # Optional: Add some basic processing so it's visible
        ply_pcd.estimate_normals()
        ply_pcd.paint_uniform_color([0.8, 0.2, 0.2]) # Muted Red
        
        print(f"Visualizing {ply_path}...")
        # Use draw_plotly for Jupyter compatibility
        o3d.visualization.draw_plotly([ply_pcd])

except Exception as e:
    print(f"An error occurred: {e}")

Visualizing 3099.ply...


In [155]:
import argparse
import torch
import numpy as np
import open3d as o3d
import copy

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud
from config import make_cfg
from model import create_model

# --- 1. SETUP & LOAD MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

WEIGHTS = "../../output/geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn/snapshots/epoch-40.pth.tar"
cfg = make_cfg()
model = create_model(cfg).cuda()
model.load_state_dict(torch.load(WEIGHTS)["model"])

# CRITICAL: Put model in evaluation mode for new data
model.eval() 

# --- 2. DEFINE PATHS ---
REF_NUM = 3
REF_FILE = f"../../data/faces/demo/ref_{REF_NUM}.npy"
SRC_FILE = "plank.npy" # Your new source data

# --- 3. INFERENCE DATA LOADER ---
def load_inference_data(src_path, ref_path):
    src_points = np.load(src_path)
    ref_points = np.load(ref_path)
    
    # GeoTransformer uses Nx1 ones for xyz coordinates when no other features exist
    src_feats = np.ones_like(src_points[:, :1])
    ref_feats = np.ones_like(ref_points[:, :1])

    data_dict = {
        "ref_points": ref_points.astype(np.float32),
        "src_points": src_points.astype(np.float32),
        "ref_feats": ref_feats.astype(np.float32),
        "src_feats": src_feats.astype(np.float32),
    }

    # Dummy variables to keep the forward pass happy
    data_dict["morphed_full"] = np.zeros_like(ref_points).astype(np.float32) 
    data_dict["gt_z"] = np.zeros((32, 100), dtype=np.float32) 
    
    # NEW: Add a dummy 4x4 identity matrix for the missing transform
    data_dict["transform"] = np.eye(4, dtype=np.float32)
    
    return data_dict

data_dict = load_inference_data(SRC_FILE, REF_FILE)

# --- 4. PREPARE DATA & INFERENCE ---
neighbor_limits = [38, 36, 36, 38]  # default setting in 3DMatch
data_dict = registration_collate_fn_stack_mode(
    [data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
)

data_dict = to_cuda(data_dict)

# Disable gradient calculation to save memory and ensure pure inference
with torch.no_grad():
    output_dict = model(data_dict)

data_dict = release_cuda(data_dict)
output_dict = release_cuda(output_dict)

# --- 5. EXTRACT RESULTS ---
ref_points = output_dict["ref_points"]
src_points = output_dict["src_points"]
estimated_transform = output_dict["estimated_transform"]

# --- 6. VISUALIZATION ---
# Create Open3D Point Clouds
ref_pcd = make_open3d_point_cloud(ref_points)
ref_pcd.estimate_normals()
ref_pcd.paint_uniform_color([0.0, 0.651, 0.929]) # Blue for Reference

src_pcd = make_open3d_point_cloud(src_points)
src_pcd.estimate_normals()
src_pcd.paint_uniform_color([1.0, 0.706, 0.0]) # Yellow for Source (plank)

# Transform the new source point cloud using the model's prediction
estimated_t_src_pcd = copy.deepcopy(src_pcd).transform(estimated_transform)


print("Visualizing: Reference (Blue) and Transformed Source/Plank (Yellow)")
o3d.visualization.draw_plotly([ref_pcd, estimated_t_src_pcd])

Visualizing: Reference (Blue) and Transformed Source/Plank (Yellow)
